In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

folders = [f for f in dbutils.fs.ls("/Volumes/dataplatform01_central_dev_catalog_01/bronze_raw_sap_bo/sapbo_webi") if f.isDir()]
for folder in folders:
    if folder.name.startswith("prod/"):
        File_format = None
        for f in dbutils.fs.ls(folder.path):
            if f.isDir() and ("object_type=csv" in f.name or "object_type=pdf" in f.name ):
                File_format = folder.name.split("object_type=")[-1].split("/")[0]
                df = (
                    spark.read.format("binaryFile")
                    .option("recursiveFileLookup", "true")
                    .load(folder.path)
                    .selectExpr("_metadata.file_path as File_location")
                    .withColumns({
                        "Document_ID": F.regexp_extract(F.col("File_location"), r"WEBI_(.+?)\.(csv|pdf)", 1),
                        "File_format": F.regexp_extract(F.col("File_location"), r"\.([^./]+)$", 1)
                    })
                    .select("Document_ID", "File_format", "File_location")
                )
            df = df.withColumns({
                "format_priority": F.when(F.col("File_format") == "csv", 1).when(F.col("File_format") == "pdf", 2).otherwise(3),
                "ingestion_ts": F.try_to_timestamp(F.regexp_extract(F.col("File_location"), r"(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2})", 1), F.lit("yyyy-MM-dd'T'HH_mm_ss"))
            })

            window = (
                Window
                .partitionBy("Document_ID")
                .orderBy(F.col("format_priority"), F.col("ingestion_ts").desc_nulls_last())
            )

            df = (
                df.select("*", F.row_number().over(window).alias("row_num"))
                .filter(F.col("row_num") == 1)
                .drop("row_num", "format_priority", "ingestion_ts")
            )
        pass

summary = (
    df.groupBy("File_format")
    .count()
    .withColumnRenamed("count", "num_files")
)
display(summary)
cms_df = spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")
cms_window = (
    Window
    .partitionBy("Document_Id", "Data_Provider_ID", "SQL_Index")
    .orderBy(F.col("ingestion_ts").desc_nulls_last())
)

cms_df = (
    cms_df
    .withColumn("row_num", F.row_number().over(cms_window))
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)
cms_df = cms_df.filter(F.col("Document_Id").isin(
    [row.Document_ID for row in df.filter(F.col("File_location").isNotNull()).select("Document_ID").distinct().collect()]
))
cms_df = cms_df.filter(F.col("File_location").isNull())
print(f"Distinct Document_Ids pending: {cms_df.select('Document_Id').distinct().count()}")
joined_df = (
    cms_df.drop("File_format", "File_location")
    .join(df.select("Document_ID", "File_format", "File_location"), cms_df.Document_Id == df.Document_ID, "left")
    .drop(df.Document_ID)
)
print(f"Rows to write back to Bronze cms table:{joined_df.count()}")
display(joined_df.select(F.count_distinct("Document_Id").alias("distinct_Document_toWrite")))
joined_df = joined_df.withColumn("ingestion_ts", F.current_timestamp())
joined_df = joined_df.filter(F.col("File_location").isNotNull())
print(f"Distinct Document_Ids to write: {joined_df.select('Document_Id').distinct().count()}")
joined_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# --- Identify Document_IDs that still have null / string-null File_location ---
scanned_records_sql = """
    select distinct Document_ID as scanned_record
    from (
        select * from (
            select *,
                rank() over (
                    partition by document_id, Data_Provider_ID, SQL_Index
                    order by ingestion_ts desc
                ) as rn
            from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms
        ) where rn = 1
    ) as c1
    where c1.document_id in (
        select distinct document_id
        from (
            select *,
                row_number() over (
                    partition by document_id
                    order by cast(ingestion_ts as timestamp) desc
                ) as rn
            from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
        )
        where rn = 1
    )
    -- cover both actual NULL and string 'null'
    and (c1.File_location is null or c1.File_location = 'null')
"""

pending_ids = {row.scanned_record for row in spark.sql(scanned_records_sql).collect()}
print(f"Document_IDs with null / string-null File_location: {len(pending_ids)}")

# --- Get latest CMS row per (Document_Id, Data_Provider_ID, SQL_Index) ---
cms_window = (
    Window
    .partitionBy("Document_Id", "Data_Provider_ID", "SQL_Index")
    .orderBy(F.col("ingestion_ts").desc_nulls_last())
)

cms_pending = (
    spark.table("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")
    .withColumn("rn", F.row_number().over(cms_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
    .filter(F.col("Document_Id").isin(pending_ids))
    # treat both actual NULL and string 'null' as missing
    .filter(F.col("File_location").isNull() | (F.col("File_location") == "null"))
    .withColumn("File_location", F.lit(None).cast("string"))  # normalise string 'null' -> actual NULL
)
print(f"CMS rows to fix: {cms_pending.count()}")
print(cms_pending.select("Document_Id").distinct().count())

cms_pending = cms_pending.withColumn("File_location", F.lit("API CSV extraction Failed"))
# cms_pending = cms_pending.withColumn("ingestion_ts", F.current_timestamp())
# display(cms_pending.select("Document_Id", "File_location", "ingestion_ts").distinct())
# cms_pending.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms")